In [12]:
import pandas as pd
import requests
import zipfile
import io
url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
response = requests.get(url)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])
display(df.head())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [13]:

print("Label distribution:\n", df['label'].value_counts())
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
display(df.head())

Label distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Shape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)

Shape of X_train_tfidf: (4457, 7702)
Shape of X_test_tfidf: (1115, 7702)


In [16]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
print("Model trained successfully!")

Model trained successfully!


In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nConfusion Matrix:\n", conf_matrix)

Accuracy: 0.9668
Precision: 1.0000
Recall: 0.7517
F1-Score: 0.8582

Confusion Matrix:
 [[966   0]
 [ 37 112]]


In [18]:
new_message = ["Congratulations! You won a free prize. Click here now!"]
new_message_tfidf = tfidf_vectorizer.transform(new_message)
prediction = model.predict(new_message_tfidf)
predicted_label = "spam" if prediction[0] == 1 else "ham"
print(f"The message: '{new_message[0]}' is predicted as: {predicted_label}")

The message: 'Congratulations! You won a free prize. Click here now!' is predicted as: spam


In [19]:

while True:
    msg = input("Enter a message (or type exit): ")
    if msg.lower() == "exit":
        break
    msg_tfidf = tfidf_vectorizer.transform([msg])
    prediction = model.predict(msg_tfidf)[0]
    label = "SPAM" if prediction == 1 else "NOT SPAM"
    print("Prediction:", label)
    print()

Enter a message (or type exit): Congratulations! You won a free prize. Click here now!
Prediction: SPAM

Enter a message (or type exit): Hey, are we meeting at the library today?
Prediction: NOT SPAM

Enter a message (or type exit): URGENT! Claim your free cash reward now!
Prediction: SPAM

Enter a message (or type exit): what are the topics for todays exam?
Prediction: NOT SPAM

Enter a message (or type exit): exit
